In [2]:
#    Define input file for BOUT++

import numpy as np
import scipy
import xbout
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
plt.rcParams["animation.html"] = "jshtml"
plt.rcParams['figure.dpi'] = 200  
plt.ioff()
#%load_ext autoreload
#%autoreload 2
params = {'legend.fontsize': 8,
'legend.title_fontsize': 8,
'figure.figsize': (4.8, 4.8),
'axes.labelsize': 12,
'axes.titlesize': 15,
'xtick.labelsize':8,
'ytick.labelsize':8,
'font.family': 'serif',
'text.usetex': False}
plt.rcParams.update(params)


analyze=True

def write_BOUT_inp(inputobj):
    # Normalization, beacuse x does indeed not go from 0 to 1, this screws up
    # the calculation for the perp derivatives
    
    minvalue=(-2.0+0.5) / (nx - 2*2)
    maxvalue=(nx-3+0.5) / (nx - 2*2)
    
    
    cont = f"""
#


nout = 200
timestep = 10

grid = "FFT_cor_Markarov_islanddivertor_260_16_1024_5.5_2.5_1.0_0.5_0.85_-0.0025_5_5.9_divertor1_pentagon_0.515_6.05_15_2.fci.grid.nc"

[mesh]
symmetricGlobalX = true

extrapolate_y = false   # Extrapolate metrics into Y boundary cells?

[mesh:paralleltransform]
type = fci

[input]
error_on_unused_options=false

[restart_files]
init_missing = true # initialize missing variables on restart?


########################################################################################


# derivative methods

[ddx]

first = C2
second = C2
upwind = W3

[ddy]

first = C4
second = C4
upwind = W3

[ddz]

first = C2
second = C2
upwind = W3


########################################################################################


[mesh:paralleltransform:xzinterpolation]
#type = hermitespline
type = monotonichermitespline


###################################################


[solver]


# Note: If evolving neutrals, need preconditioning
type = snes
snes_type=anderson
#use_precon = true

atol = 1.0e-10  # absolute tolerance
rtol = 1.0e-5   # relative tolerance
mxstep = 8000  # Maximum internal steps per output
pctype=hypre
matrix_free=true


#cvode_max_order = 2
#cvode_stability_limit_detection = true


########################################################################################


[phiSolver]

type   = petsc  # Needed if Boussinesq = false

ksptype = gmres # Linear iterative method

pctype = hypre # Preconditioner. Direct "lu" or "ilu"; iterative "jacobi", "sor"

atol=1e-8
rtol = 1e-5
# Set package to perform factorisation for direct solves
# "petsc" built-in solver only serial
# "superlu", "superlu_dist", "mumps", "cusparse"
factor_package = superlu_dist

inner_boundary_flags = 2
outer_boundary_flags = 16

all_terms = false
nonuniform = true   # NOTE: Necessary to avoid numerical instability


#########################################################################################


[aparSolver]
type = petsc

rtol=1e-5
atol=1e-8

# ksptype = gmres   # Linear iterative method

# pctype  = hypre   # Preconditioner. Direct "lu" or "ilu"; iterative "jacobi", "sor"

# # Set package to perform factorisation for direct solves
# # "petsc" built-in solver only serial
# # "superlu", "superlu_dist", "mumps", "cusparse"
# factor_package = superlu_dist


inner_boundary_flags = 2
outer_boundary_flags = 2

# all_terms = true
nonuniform = true

# general settings for the model
[input]

transform_from_field_aligned=False


########################################################################################


[Hermes]

electromagnetic = false
j_pol_pi = false
j_pol_simplified = true
boussinesq = true
FiniteElMass = true


evolve_ne = {inputobj.Ne["evolve_ne"]}
evolve_nvi = {inputobj.NVi["evolve_nvi"]}
evolve_te = {inputobj.Pe["evolve_pe"]}
evolve_ti = {inputobj.Pi["evolve_pi"]}

evolve_vort = {inputobj.Vort["evolve_vort"]}
evolve_vepsi = {inputobj.VePsi["evolve_vepsi"]}


TE_Ne = false
TE_NVi = false
TE_Pe = false
TE_Pi = false
TE_Vort = false
TE_VePsi = false
verbose = false
output_ddt = false

calc_potential = 1

Tnorm = 20
Nnorm = 1e19
Bnorm = 1.0
AA = 2.0


########################################################################################


[Transportcoefficients]

anomalous_D = {inputobj.Transportcoefficients["anomalous_D"]}
anomalous_nu = {inputobj.Transportcoefficients["anomalous_nu"]}
anomalous_chi = {inputobj.Transportcoefficients["anomalous_chi"]}

sigma=0.03

hyper_D = {inputobj.Transportcoefficients["hyper_D"]}
hyper_nu = {inputobj.Transportcoefficients["hyper_nu"]}
hyper_chi = {inputobj.Transportcoefficients["hyper_chi"]}


num_D = {inputobj.Transportcoefficients["num_D"]}
num_nu = {inputobj.Transportcoefficients["num_nu"]}
num_chi = {inputobj.Transportcoefficients["num_chi"]}


########################################################################################


[Numerics]


use_Div_n_bxGrad_f_B_XPPM = false
use_bracket = true
use_new_conduction = false
use_new_viscosity = false
use_new_div_par = false

check_finite = false

ne_bndry_flux = true
pe_bndry_flux = true
vort_bndry_flux = true

flux_limit_alpha = -1
kappa_limit_alpha = 0.3
eta_limit_alpha = -0.5

resistivity_multiply = 10.0
electron_weight = 1.0
scale_ExB = 1.0

floor_Ne = 0.05
floor_Te = 1.0/20.0
floor_Ti = 1.0/20.0

use_viscosity_limiter = false
viscosity_limiter_value = 20.0

NVi_supsonic_factor = 1e-5
Ve_supsonic_factor = 5


########################################################################################


[Sheath]

sheath_model = 0
par_sheath_model = 0
sheath_gamma_e = 5.0
sheath_gamma_i = 3.5

parallel_sheaths = true
sheath_allow_supersonic = true
sheath_infsink = false
infsink_Te = 0.5


########################################################################################
########################################################################################
########################################################################################


[all]


bndry_par_all = parallel_neumann_o1


########################################################################################


[Ne] # Electron density
bndry_xin = dirichlet(Ne:solution)
bndry_xout =dirichlet(Ne:solution)
bndry_par = parallel_neumann_o2


use_Vi = true

Ne_ExB = {inputobj.Ne["Ne_ExB"]}
Ne_mag = {inputobj.Ne["Ne_mag"]}
Ne_parflow = {inputobj.Ne["Ne_parflow"]}
Ne_collision = {inputobj.Ne["Ne_collision"]}
Ne_anomalous = {inputobj.Ne["Ne_anomalous"]}
Ne_sources = {inputobj.Ne["Ne_sources"]}
Ne_hyper = {inputobj.Ne["Ne_hyper"]}
Ne_numdiff = {inputobj.Ne["Ne_numdiff"]}


########################################################################################


[NVi]
bndry_xin = dirichlet(NVi:solution)
bndry_xout =dirichlet(NVi:solution)



NVi_ExB = {inputobj.NVi["NVi_ExB"]}
NVi_mag = {inputobj.NVi["NVi_mag"]}
NVi_parflow = {inputobj.NVi["NVi_parflow"]}
NVi_parpressure = {inputobj.NVi["NVi_parpressure"]}
NVi_parviscos = {inputobj.NVi["NVi_parviscos"]}
NVi_collision = {inputobj.NVi["NVi_collision"]}
NVi_anomalous = {inputobj.NVi["NVi_anomalous"]}
NVi_hyper = {inputobj.NVi["NVi_hyper"]}
NVi_numdiff = {inputobj.NVi["NVi_numdiff"]}
NVi_supsonicdampening = {inputobj.NVi["NVi_supsonicdampening"]}


########################################################################################


[Pe]  # Electron pressure

bndry_xin = dirichlet(Pe:solution)
bndry_xout =dirichlet(Pe:solution)


Pe_ExB = {inputobj.Pe["Pe_ExB"]}
Pe_mag = {inputobj.Pe["PePe_mag_ExB"]}
Pe_parflow = {inputobj.Pe["Pe_parflow"]}
Pe_conduction = {inputobj.Pe["Pe_conduction"]}
Pe_ohmic = {inputobj.Pe["Pe_ohmic"]}
Pe_thermalforce = {inputobj.Pe["Pe_thermalforce"]}
Pe_thermalcurrent = {inputobj.Pe["Pe_thermalcurrent"]}
Pe_collision = {inputobj.Pe["Pe_collision"]}
Pe_anomalous = {inputobj.Pe["Pe_anomalous"]}
Pe_sources = {inputobj.Pe["Pe_sources"]}
Pe_energyexchange = {inputobj.Pe["Pe_energyexchange"]}
Pe_hyper = {inputobj.Pe["Pe_hyper"]}
Pe_numdiff = {inputobj.Pe["Pe_numdiff"]}
Pe_dampening = {inputobj.Pe["Pe_dampening"]}


########################################################################################


[Pi]
bndry_xin = dirichlet(Pi:solution)
bndry_xout =dirichlet(Pi:solution)


Pi_ExB = {inputobj.Pi["Pi_ExB"]}
Pi_mag = {inputobj.Pi["Pi_mag"]}
Pi_parflow = {inputobj.Pi["Pi_parflow"]}
Pi_conduction = {inputobj.Pi["Pi_conduction"]}
Pi_diamagenergyexchange = {inputobj.Pi["Pi_diamagenergyexchange"]}
Pi_parviscousheat = {inputobj.Pi["Pi_parviscousheat"]}
Pi_resistivedrift = {inputobj.Pi["Pi_resistivedrift"]}
Pi_perpviscous = {inputobj.Pi["Pi_perpviscous"]}
Pi_sources = {inputobj.Pi["Pi_sources"]}
Pi_hyper = {inputobj.Pi["Pi_hyper"]}
Pi_numdiff = {inputobj.Pi["Pi_numdiff"]}
Pi_anomalous = {inputobj.Pi["Pi_anomalous"]}
Pi_energyexchange = {inputobj.Pi["Pi_energyexchange"]}


########################################################################################


[Vort]
bndry_xin = dirichlet(Vort:solution)
bndry_xout =dirichlet(Vort:solution)


Vort_mag = {inputobj.Vort["Vort_mag"]}
Vort_parcurrent = {inputobj.Vort["Vort_parcurrent"]}
Vort_polarcurrent = {inputobj.Vort["Vort_polarcurrent"]}
Vort_collision = {inputobj.Vort["Vort_collision"]}
Vort_parviscous = {inputobj.Vort["Vort_parviscous"]}
Vort_anomalous = {inputobj.Vort["Vort_anomalous"]}
Vort_hyper = {inputobj.Vort["Vort_hyper"]}
Vort_numdiff = {inputobj.Vort["Vort_numdiff"]}
Vort_parflow = {inputobj.Vort["Vort_parflow"]}


########################################################################################


[VePsi] 
bndry_xin = dirichlet(VePsi:solution)
bndry_xout =dirichlet(VePsi:solution)


VePsi_parefield = {inputobj.VePsi["VePsi_parefield"]}
VePsi_parpressure = {inputobj.VePsi["VePsi_parpressure"]}
VePsi_partemp = {inputobj.VePsi["VePsi_partemp"]}
VePsi_parcurrent = {inputobj.VePsi["VePsi_parcurrent"]}
VePsi_ExB = {inputobj.VePsi["VePsi_ExB"]}
VePsi_parflow = {inputobj.VePsi["VePsi_parflow"]}
VePsi_hyper = {inputobj.VePsi["VePsi_hyper"]}
VePsi_numdiff = {inputobj.VePsi["VePsi_numdiff"]}
VePsi_parallelvisc = {inputobj.VePsi["VePsi_parallelvisc"]}
VePsi_supsonicdampening = {inputobj.VePsi["VePsi_supsonicdampening"]}
VePsi_anomalous = {inputobj.VePsi["VePsi_anomalous"]}





    """
    return cont

def write_slurmfile(id,nodes,ntasks_per_node, mem,time_h,time_min,
                   executable,directory):
    content=f"""#!/bin/bash -l                                                                                                                                        
# Standard output and error:                                                                                                                          
#SBATCH -o ./job_hybrid.out.%j                                                                                                                        
#SBATCH -e ./job_hybrid.err.%j                                                                                                                        
# Initial working directory:                                                                                                                          
#SBATCH -D ./                                                                                                                                         
# Job Name:                                                                                                                                           
#SBATCH -J MMS_reduced_MHD_{int(id)}                                                                                                                         
#                                                                                                                                                     
# Number of nodes and MPI tasks per node:                                                                                                             
#SBATCH --nodes={nodes}                                                                                                                                    
#SBATCH --ntasks-per-node={int(ntasks_per_node)}                                                                                                                          
#SBATCH --mem={int(mem)}GB                                                                                                                                    

# for OpenMP:                                                                                                                                         
#SBATCH --cpus-per-task=1                                                                                                                             
#SBATCH --mail-type=all                                                                                                                               
#SBATCH --mail-user=toto@ipp.mpg.de                                                                                                                   
#                                                                                                                                                     
# Wall clock limit (max. is 24 hours):                                                                                                                
#SBATCH --time={int(time_h)}:{int(time_min)}:00                                                                                                                               
set -x
# Load compiler and MPI modules (must be the same as used for compiling the code)                                                                     
module purge
module load gcc/13 openmpi/4.1 hdf5-serial/1.14.1 netcdf-serial/4.9.2 fftw-mpi/3.3.10 anaconda/3/2021.11 mkl/2024.0

export OMP_NUM_THREADS=$SLURM_CPUS_PER_TASK

# For pinning threads correctly:                                                                                                                      
export OMP_PLACES=cores

# Run the program                                                                                                                                     

srun {executable} -d {directory} $@
    
    """
    return content


def gridfile_string(nx,ny,nz,rho_1,xmin,xmax,Ly,B0,q0,q1,directory):
    content = f"{directory}/cor_mms_Axial_Circle_{nx}_{ny}_{nz}_{rho_1}_{xmin}_{xmax}_{np.round(Ly,3)}_{B0}_{q0}_{q1}.fci.grid.nc"
    return content

def read_solution_source(var):
    with open(f"solution_{var}.txt", "r") as file:
        solution = file.read()
        
    with open(f"source_{var}.txt", "r") as file:
        source = file.read()
    
    return solution,source


class Input_class():
    def __init(self):
        self.Vort={}
        self.Ne={}
        self.NVi={}
        self.Pe={}
        self.Pi={}
        self.VePsi={}
        self.Transportcoefficients={}
    
    def set_Vort(evolve_vort , Vort_mag , Vort_parcurrent , Vort_polarcurrent , Vort_collision , Vort_parviscous , 
                Vort_anomalous , Vort_hyper , Vort_numdiff , Vort_parflow):
        self.Vort["evolve_vort"]=evolve_vort
        self.Vort["Vort_mag"]=Vort_mag
        self.Vort["Vort_parcurrent"]=Vort_parcurrent
        self.Vort["Vort_polarcurrent"]=Vort_polarcurrent
        self.Vort["Vort_collision"]=Vort_collision
        self.Vort["Vort_parviscous"]=Vort_parviscous
        self.Vort["evolve_VVort_anomalousort"]=Vort_anomalous
        self.Vort["Vort_hyper"]=Vort_hyper
        self.Vort["Vort_numdiff"]=Vort_numdiff
        self.Vort["Vort_parflow"]=Vort_parflow
        
    def set_VePsi(self, evolve_vepsi, VePsi_parefield, VePsi_parpressure, VePsi_partemp, VePsi_parcurrent, VePsi_ExB, 
              VePsi_parflow, VePsi_hyper, VePsi_numdiff, VePsi_parallelvisc, VePsi_supsonicdampening, 
              VePsi_anomalous):
        self.VePsi["evolve_vepsi"] = evolve_vepsi
        self.VePsi["VePsi_parefield"] = VePsi_parefield
        self.VePsi["VePsi_parpressure"] = VePsi_parpressure
        self.VePsi["VePsi_partemp"] = VePsi_partemp
        self.VePsi["VePsi_parcurrent"] = VePsi_parcurrent
        self.VePsi["VePsi_ExB"] = VePsi_ExB
        self.VePsi["VePsi_parflow"] = VePsi_parflow
        self.VePsi["VePsi_hyper"] = VePsi_hyper
        self.VePsi["VePsi_numdiff"] = VePsi_numdiff
        self.VePsi["VePsi_parallelvisc"] = VePsi_parallelvisc
        self.VePsi["VePsi_supsonicdampening"] = VePsi_supsonicdampening
        self.VePsi["VePsi_anomalous"] = VePsi_anomalous
    
    def set_Pi(self, evolve_pi, Pi_ExB, Pi_mag, Pi_parflow, Pi_conduction, Pi_diamagenergyexchange, 
           Pi_parviscousheat, Pi_resistivedrift, Pi_perpviscous, Pi_sources, Pi_hyper, Pi_numdiff, 
           Pi_anomalous, Pi_energyexchange):
        self.Pi["evolve_pi"] = evolve_pi
        self.Pi["Pi_ExB"] = Pi_ExB
        self.Pi["Pi_mag"] = Pi_mag
        self.Pi["Pi_parflow"] = Pi_parflow
        self.Pi["Pi_conduction"] = Pi_conduction
        self.Pi["Pi_diamagenergyexchange"] = Pi_diamagenergyexchange
        self.Pi["Pi_parviscousheat"] = Pi_parviscousheat
        self.Pi["Pi_resistivedrift"] = Pi_resistivedrift
        self.Pi["Pi_perpviscous"] = Pi_perpviscous
        self.Pi["Pi_sources"] = Pi_sources
        self.Pi["Pi_hyper"] = Pi_hyper
        self.Pi["Pi_numdiff"] = Pi_numdiff
        self.Pi["Pi_anomalous"] = Pi_anomalous
        self.Pi["Pi_energyexchange"] = Pi_energyexchange


    def set_Pe(self, evolve_pe, Pe_ExB, Pe_mag, Pe_parflow, Pe_conduction, Pe_ohmic, Pe_thermalforce, 
           Pe_thermalcurrent, Pe_collision, Pe_anomalous, Pe_sources, Pe_energyexchange, Pe_hyper, 
           Pe_numdiff, Pe_dampening):
        self.Pe["evolve_pe"] = evolve_pe
        self.Pe["Pe_ExB"] = Pe_ExB
        self.Pe["Pe_mag"] = Pe_mag
        self.Pe["Pe_parflow"] = Pe_parflow
        self.Pe["Pe_conduction"] = Pe_conduction
        self.Pe["Pe_ohmic"] = Pe_ohmic
        self.Pe["Pe_thermalforce"] = Pe_thermalforce
        self.Pe["Pe_thermalcurrent"] = Pe_thermalcurrent
        self.Pe["Pe_collision"] = Pe_collision
        self.Pe["Pe_anomalous"] = Pe_anomalous
        self.Pe["Pe_sources"] = Pe_sources
        self.Pe["Pe_energyexchange"] = Pe_energyexchange
        self.Pe["Pe_hyper"] = Pe_hyper
        self.Pe["Pe_numdiff"] = Pe_numdiff
        self.Pe["Pe_dampening"] = Pe_dampening

    def set_NVi(self, evolve_nvi, NVi_ExB, NVi_mag, NVi_parflow, NVi_parpressure, NVi_parviscos, NVi_collision, 
            NVi_anomalous, NVi_hyper, NVi_numdiff, NVi_supsonicdampening):
        self.NVi["evolve_nvi"] = evolve_nvi
        self.NVi["NVi_ExB"] = NVi_ExB
        self.NVi["NVi_mag"] = NVi_mag
        self.NVi["NVi_parflow"] = NVi_parflow
        self.NVi["NVi_parpressure"] = NVi_parpressure
        self.NVi["NVi_parviscos"] = NVi_parviscos
        self.NVi["NVi_collision"] = NVi_collision
        self.NVi["NVi_anomalous"] = NVi_anomalous
        self.NVi["NVi_hyper"] = NVi_hyper
        self.NVi["NVi_numdiff"] = NVi_numdiff
        self.NVi["NVi_supsonicdampening"] = NVi_supsonicdampening
        
    
    def set_Ne(self, evolve_ne, Ne_ExB, Ne_mag, Ne_parflow, Ne_collision, Ne_anomalous, Ne_sources, Ne_hyper, Ne_numdiff):
        self.Ne["evolve_ne"] = evolve_ne
        self.Ne["Ne_ExB"] = Ne_ExB
        self.Ne["Ne_mag"] = Ne_mag
        self.Ne["Ne_parflow"] = Ne_parflow
        self.Ne["Ne_collision"] = Ne_collision
        self.Ne["Ne_anomalous"] = Ne_anomalous
        self.Ne["Ne_sources"] = Ne_sources
        self.Ne["Ne_hyper"] = Ne_hyper
        self.Ne["Ne_numdiff"] = Ne_numdiff

    def set_Transportcoefficients(self,anomalous_D,anomalous_nu,anomalous_chi,
                                  hyper_D,hyper_nu,hyper_chi,
                                  num_D,num_nu,num_chi):
        self.Transportcoefficients["anomalous_D"]=anomalous_D
        self.Transportcoefficients["anomalous_nu"]=anomalous_nu
        self.Transportcoefficients["anomalous_chi"]=anomalous_chi
        self.Transportcoefficients["hyper_D"]=hyper_D
        self.Transportcoefficients["hyper_nu"]=hyper_nu
        self.Transportcoefficients["hyper_chi"]=hyper_chi
        self.Transportcoefficients["num_D"]=num_D
        self.Transportcoefficients["num_nu"]=num_nu
        self.Transportcoefficients["num_chi"]=num_chi
        
        
        
